# GPSseq · Environment Compatibility Setup

This notebook completes the **GPSseq-specific TensorFlow / Neurite / VoxelMorph setup** after `requirements.txt` has been installed.

> **Run this notebook after `python -m pip install -r requirements.txt`.**
>
> GPSseq does **not** use the public PyPI `voxelmorph==0.2` package directly. It uses a
> validated historical VoxelMorph source snapshot bundled in
> `third_party/voxelmorph-gpsseq-legacy.zip`, because the GPSseq affine model requires
> `VxmAffineFeatureDetector`.

### Installation order

1. Check the active Python environment.
2. Install the GPSseq-compatible VoxelMorph source snapshot.
3. Patch two legacy Neurite Keras imports for TensorFlow 2.16 / Keras 3.
4. Verify Neurite and VoxelMorph in a **fresh Python process**.
5. Verify that `VxmAffineFeatureDetector` can be instantiated.

After this notebook succeeds, continue with:

`tutorials/02_prepare_allen_atlas.ipynb`


## 1. Check the active environment

The notebook assumes that the conda environment has already been created and that
`requirements.txt` has already been installed.

The validated development environment uses:

- Python 3.9
- TensorFlow 2.16.1
- Keras 3.9.0
- Neurite 0.2


In [ ]:
from pathlib import Path
import importlib.metadata as md
import subprocess
import sys
import os

print("Python:", sys.version)
print("Executable:", sys.executable)

# Locate the GPSseq repository root without assuming the notebook's working directory.
def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    candidates = [start, *start.parents]
    for p in candidates:
        if (
            (p / "requirements.txt").is_file()
            and (p / "brain_registration").is_dir()
            and (p / "third_party").is_dir()
        ):
            return p
    raise FileNotFoundError(
        "Could not locate the GPSseq repository root. "
        "Open this notebook from inside the GPSseq repository."
    )

ROOT = find_repo_root()
VXM_ARCHIVE = ROOT / "third_party" / "voxelmorph-gpsseq-legacy.zip"
PATCH_SCRIPT = ROOT / "scripts" / "patch_neurite.py"
VERIFY_SCRIPT = ROOT / "scripts" / "verify_installation.py"

print("GPSseq root:", ROOT)
print("VoxelMorph archive:", VXM_ARCHIVE)
print("Neurite patch script:", PATCH_SCRIPT)
print("Verification script:", VERIFY_SCRIPT)

if not VXM_ARCHIVE.is_file():
    raise FileNotFoundError(f"Missing: {VXM_ARCHIVE}")
if not PATCH_SCRIPT.is_file():
    raise FileNotFoundError(f"Missing: {PATCH_SCRIPT}")
if not VERIFY_SCRIPT.is_file():
    raise FileNotFoundError(f"Missing: {VERIFY_SCRIPT}")

# Check installed distributions without importing TensorFlow/Neurite/VoxelMorph yet.
for package in ("tensorflow", "keras", "neurite"):
    try:
        print(f"{package}: {md.version(package)}")
    except md.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")


## 2. Install the GPSseq-compatible VoxelMorph source

**Do not run `pip install voxelmorph` here.**

The public PyPI `voxelmorph==0.2` release is not the same source snapshot used by GPSseq.
The bundled archive contains the TensorFlow implementation required by GPSseq,
including `VxmAffineFeatureDetector`.

`--no-deps` is intentional: the dependencies were already installed from
`requirements.txt`, and this step should replace only the VoxelMorph package itself.


In [ ]:
cmd = [
    sys.executable,
    "-m",
    "pip",
    "install",
    "--no-deps",
    "--force-reinstall",
    str(VXM_ARCHIVE),
]

print("Running:")
print(" ".join(f'"{x}"' if " " in x else x for x in cmd))

subprocess.run(cmd, check=True)
print("\nVoxelMorph source installation completed.")


## 3. Patch Neurite for TensorFlow 2.16 / Keras 3

Neurite 0.2 contains legacy imports such as:

```python
from tensorflow.keras.losses import mean_absolute_error as l1
from tensorflow.keras.losses import mean_squared_error as l2
```

With the validated GPSseq environment, these imports are replaced by the Keras 3
aliases used by the working environment:

```python
from tensorflow.keras.losses import MAE as l1
from tensorflow.keras.losses import MSE as l2
```

The patch script edits only the installed Neurite files:

- `neurite/tf/metrics.py`
- `neurite/tf/losses.py`

A backup is created before modification.


In [ ]:
cmd = [
    sys.executable,
    str(PATCH_SCRIPT),
    "--apply",
]

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)


## 4. Verify the installation

The verification is executed in a **fresh Python process** so that it does not depend
on packages already imported into this notebook's kernel.

The check confirms:

- TensorFlow / Keras / Neurite / VoxelMorph are installed.
- Neurite imports successfully.
- VoxelMorph imports successfully.
- `VxmAffineFeatureDetector` exists.
- The affine network can be instantiated with the GPSseq input shape `(256, 256)`.


In [ ]:
subprocess.run(
    [sys.executable, str(VERIFY_SCRIPT)],
    check=True,
)


## 5. Final result

If the previous cell finishes without an error, the environment is ready for GPSseq.

You can now continue to:

```text
tutorials/02_prepare_allen_atlas.ipynb
```

That notebook downloads the public Allen Mouse Brain Atlas files and prepares the
required local directory structure.

> **Important:** if you later reinstall or upgrade `neurite`, the compatibility patch
> may be overwritten. Re-run this installation notebook after such a change.
